# Building Models

Networks are built by chaining layers. The chain operator `(~~>)` threads dimensions through the type — the output of each layer must match the input of the next, checked at compile time.

Two related types from tutorial 01:
- `Array [dims] ty` — pure-Idris structural data (Vect-of-Vect with `Functor`/`Num`/`Floating` instances). What you build training data with.
- `Tensor [dims] d` — autograd-tracked tensor on a backend (CPU / CUDA / MPS). What weights and forward outputs are.

## Layer constructors

Each `*LayerAny` constructor builds a layer wrapped in the `AnyLayer` existential, ready to chain. The string argument is the **paramPrefix** — every learnable parameter the layer creates is registered in the C-side optimizer registry under that prefix.

In [ ]:
:t linearLayerAny


In [ ]:
:t reluLayerAny


## Composition with `(~~>)`

Build a one-hidden-layer MLP. Each layer is given a distinct paramPrefix:

In [ ]:
:exec do { srand 42;
  l1 <- linearLayerAny {i=2} {o=8} "l1";
  l2 <- linearLayerAny {i=8} {o=3} "l2";
  model <- pure (l1 ~~> reluLayerAny ~~> OutputLayer l2);
  putStrLn "Model built." }


## Shape mismatches are compile errors

If `l1`'s output dimension doesn't match `l2`'s input, the compiler rejects the chain. No runtime crash, no silent broadcast.

The following snippet is **expected to fail compilation** because `l1` outputs 8 dims but `l2` expects 5:

```idris
:exec do { srand 42;
  l1 <- linearLayerAny {i=4} {o=8} "l1";
  l2 <- linearLayerAny {i=5} {o=3} "l2";
  model <- pure (l1 ~~> OutputLayer l2);  -- compile error
  putStrLn "should not reach here" }
```

Idris reports `Mismatch between: 8 and 5.` at compile time. PyTorch would give a runtime crash.


## Forward pass

`forwardVar` walks the network, threading a `Tensor [i] d` input through each layer to produce a `Tensor [o] d` output. The autograd graph is built as a side effect on the C-side tape and consumed later by `nativeTrainStep`.

In [ ]:
:t forwardVar


In [ ]:
:exec do { srand 42;
  ll <- linearLayerAny {i=2} {o=3} "ll";
  model <- pure (the (Network 2 [] 3 CPU) (OutputLayer ll));
  let inT = the (TVec 2 CPU) (MkTensor (bulkToTensor (the (Vector 2 Double) (VArray [1.0, 2.0]))) Nothing);
  let (_, outT) = forwardVar model inT;
  putStrLn ("Output[0] = " ++ show (prim__item1d outT.tensorPtr 0));
  putStrLn ("Output[1] = " ++ show (prim__item1d outT.tensorPtr 1));
  putStrLn ("Output[2] = " ++ show (prim__item1d outT.tensorPtr 2)) }


## Debugging: `forwardVarTraced`

Drop-in replacement for `forwardVar` that prints per-layer min / max / mean / NaN-check to stderr as the forward pass runs. The autograd graph is preserved; the reductions create non-grad-tracking tape entries that get released at the next `tape_reset`.

In [ ]:
:exec do { srand 42;
  l1 <- linearLayerAny {i=4} {o=8} "l1";
  l2 <- linearLayerAny {i=8} {o=2} "l2";
  model <- pure (the (Network 4 [8, 8] 2 CPU) (l1 ~~> reluLayerAny ~~> OutputLayer l2));
  let inT = the (TVec 4 CPU) (MkTensor (bulkToTensor (the (Vector 4 Double) (VArray [0.5, -0.3, 0.1, 0.7]))) Nothing);
  _ <- forwardVarTraced "trace" model inT;
  pure () }


## Available layers

All layer constructors follow the same `*LayerAny` shape: take dim implicits + paramPrefix, return `AnyLayer i o CPU` ready to chain. Activation layers (`reluLayerAny`, `tanhLayerAny`, `sigmoidLayerAny`, `geluLayerAny`, `siluLayerAny`) are stateless `AnyLayer n n CPU` values — no IO, no paramPrefix.

In [ ]:
:t rnnLayerAny


In [ ]:
:t lstmLayerAny


In [ ]:
:t gruLayerAny


In [ ]:
:t conv2dLayerAny


In [ ]:
:t dropoutLayerAny


In [ ]:
:t embeddingLayerAny


In [ ]:
:t tanhLayerAny


In [ ]:
:t sigmoidLayerAny


## Multi-network paramId scoping

For multi-network examples (A2C / PPO / SAC actor + critic, DQN online + target), pick distinct prefixes per network and use `nativeAdamGroup "actor_" ...` to scope the optimizer to one network's parameters. This is the V2 replacement for V1's `autoNameScoped` workaround.

In [ ]:
:exec do { srand 42;
  -- Two networks under distinct paramId scopes:
  actor1 <- linearLayerAny {i=4} {o=8} "actor_l1";
  actor2 <- linearLayerAny {i=8} {o=2} "actor_l2";
  critic1 <- linearLayerAny {i=4} {o=8} "critic_l1";
  critic2 <- linearLayerAny {i=8} {o=1} "critic_l2";
  let actor = the (Network 4 [8] 2 CPU) (actor1 ~~> OutputLayer actor2);
  let critic = the (Network 4 [8] 1 CPU) (critic1 ~~> OutputLayer critic2);
  putStrLn "Both networks built; their parameters are registered under distinct prefixes." }


## Discovering the API

Use `:browse` to list everything exported from a module:

In [ ]:
:browse Layer.Linear


Next: [03 Data and Loss](03_data_and_loss.ipynb) — creating training data and computing loss with type-checked dimensions.